# 07 — Z-scored THINGS category embeddings from exemplar `.npy` trees (valid129 / valid85)

> **Stage 0b** — Run after notebook **06** (or `exemplar_set_zscore_embeddings.py`). See [`00_build_exemplar_embeddings.md`](00_build_exemplar_embeddings.md). BabyDINOv3 THINGS: scripts in step 0c–0d of that doc.

Mirror of **06** for **THINGS** (`things_bv_overlapping_categories_corrected`): layout `{root}/{category}/**/*.npy` for CLIP and DINOv3. THINGS CLIP trees use ``embedding_*.npy`` while DINOv3 uses ``{category}_*.npy``, so we **cannot** pair by basename like BV. We **L2-normalize** each vector, take the **mean over all ``*.npy`` in that category per model** (CLIP and DINOv3 separately), keep only categories with **at least one** exemplar in **both** trees, then **z-score** each dimension **across categories** (within the selected set only).

**valid129:** `data/included_categories_valid129.txt` — all paired THINGS exemplars per listed category.

**valid85:** Same **category set** as BV **06** / **03** (`included_categories_valid85.txt` ∩ per-class precision ∩ sampled manifest ∩ per-file precision). Averaging uses **all** paired THINGS exemplars in each category (THINGS images do not share BV crop stems, so we do not intersect BV’s CLIP path list here).

**CLIP filtering:** Use the on-disk THINGS CLIP tree (typically exports already restricted by CLIP threshold, e.g. `clip_image_embeddings_npy_by_category` alongside `clip_image_embeddings_doc_normalized_filtered-by-clip-*`).

**Setup:** set `CATEGORY_SETS` in the setup cell (default `['valid129']`; use `['valid85']` or `['valid129', 'valid85']` as needed). Other env vars remain: `THINGS_CLIP_NPY_DIR`, `THINGS_DINOV3_EMBEDDINGS_DIR`, `BV_PRECISION_THRESHOLD`, optional `THINGS_CROP_PATH_PREFIX` / `THINGS_CROP_PATH_PREFIX_NEW` for `.npy` path remaps.

**Outputs:** `exemplar_set_embeddings/{valid129|valid85}/things_{clip|dinov3}_exemplar_avg_{zscore,raw}_within_{set}.csv` and `things_exemplar_embedding_run.json`.

**BabyDINOv3 THINGS:** per-image embeddings from `create_babydinov3_things_embeddings.py` (default tree: `.../babydinov3_grad_accum_1/step_{BV_BABYDINOV3_CHECKPOINT_STEP}/`), then z-score via `things_exemplar_set_zscore_babydinov3.py` → `things_babydinov3_exemplar_avg_*_within_{set}.csv`.

**Run:** from `analysis/manuscript-2026/`. Requires `pandas`, `numpy` only.

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve project root from this notebook location: analysis/manuscript-2026/
PROJECT_ROOT = Path.cwd().resolve().parents[1]
DATA_DIR = PROJECT_ROOT / "data"
ANNOTATION_DIR = PROJECT_ROOT / "annotation"
PREPRINT_DIR = PROJECT_ROOT / "analysis" / "manuscript-2026"
OUT_ROOT = PREPRINT_DIR / "exemplar_set_embeddings"

CATEGORY_FILES = {
    "valid85": DATA_DIR / "included_categories_valid85.txt",
    "valid129": DATA_DIR / "included_categories_valid129.txt",
}
PER_CLASS_PRECISION_CSV = ANNOTATION_DIR / "per_class_validation_data.csv"
PER_FILE_PRECISION_CSV = ANNOTATION_DIR / "per_file_precision_data.csv"
SAMPLED_EXEMPLAR_CSV = ANNOTATION_DIR / (
    "sampled_object_crops_100_bucket_assignments_100ex_8subj_per_video_cap_babyview_only.csv"
)
PRECISION_THRESHOLD = float(os.environ.get("BV_PRECISION_THRESHOLD", "0.6"))
CROP_PREFIX = os.environ.get("THINGS_CROP_PATH_PREFIX", "").strip()
CROP_PREFIX_NEW = os.environ.get("THINGS_CROP_PATH_PREFIX_NEW", "").strip()

THINGS_CLIP_NPY_DIR = Path(
    os.environ.get(
        "THINGS_CLIP_NPY_DIR",
        "/ccn2/dataset/babyview/outputs_20250312/things_image_embeddings/clip_embeddings",
    )
).expanduser()
THINGS_DINOV3_DIR = Path(
    os.environ.get(
        "THINGS_DINOV3_EMBEDDINGS_DIR",
        str(
            Path(os.environ.get("THINGS_IMAGE_EMBEDDINGS_BASE", "/ccn2/dataset/babyview/outputs_20250312"))
            / "image_embeddings"
            / "things_bv_overlapping_categories_corrected"
            / "facebook_dinov3-vitb16-pretrain-lvd1689m"
        ),
    )
).expanduser()

# Set once at the top of notebook.
# Defaults to valid129 only; for both runs use: ["valid129", "valid85"]
CATEGORY_SETS = ["valid129"]
for s in CATEGORY_SETS:
    if s not in CATEGORY_FILES:
        raise ValueError(f"Unknown CATEGORY_SETS component {s!r}")

print(f"[07_things_exemplar_set_zscore_embeddings] CATEGORY_SETS={CATEGORY_SETS!r}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Preprint dir: {PREPRINT_DIR}")
print(f"Precision threshold: {PRECISION_THRESHOLD}")
print(f"THINGS_CLIP_NPY_DIR: {THINGS_CLIP_NPY_DIR} (exists: {THINGS_CLIP_NPY_DIR.is_dir()})")
print(f"THINGS_DINOV3_DIR: {THINGS_DINOV3_DIR} (exists: {THINGS_DINOV3_DIR.is_dir()})")
print(f"OUT_ROOT: {OUT_ROOT}")


PROJECT_ROOT /home/j7yang/babyview-projects/vss2026/object-detection
CATEGORY_SETS ['valid129', 'valid85']
PRECISION_THRESHOLD 0.6
THINGS_CLIP_NPY_DIR /ccn2/dataset/babyview/outputs_20250312/things_image_embeddings/clip_embeddings exists: True
THINGS_DINOV3_DIR /ccn2/dataset/babyview/outputs_20250312/image_embeddings/things_bv_overlapping_categories_corrected/facebook_dinov3-vitb16-pretrain-lvd1689m exists: True
OUT_ROOT /home/j7yang/babyview-projects/vss2026/object-detection/analysis/manuscript-2026/exemplar_set_embeddings


In [2]:
def resolve_category_subdir(embed_root: Path, cat: str) -> Path | None:
    direct = embed_root / cat
    if direct.is_dir():
        return direct
    for p in embed_root.iterdir():
        if p.is_dir() and p.name.lower() == cat.lower():
            return p
    return None


def load_included_categories(txt_path: Path) -> list[str]:
    return [line.strip().lower() for line in txt_path.read_text().splitlines() if line.strip()]


def load_valid_classes(per_class_csv: Path, threshold: float) -> set[str]:
    df = pd.read_csv(per_class_csv, usecols=["class", "precision"])
    df["class"] = df["class"].astype(str).str.strip().str.lower()
    return set(df.loc[df["precision"] > threshold, "class"])


def load_valid_pairs(per_file_csv: Path, threshold: float) -> set[tuple[str, str]]:
    df = pd.read_csv(per_file_csv, usecols=["filename", "class", "precision"])
    df = df[df["precision"] > threshold].copy()
    df["class_norm"] = df["class"].astype(str).str.strip().str.lower()
    df["stem"] = (
        df["filename"]
        .astype(str)
        .str.strip()
        .str.rsplit("/", n=1)
        .str[-1]
        .str.rsplit(".", n=1)
        .str[0]
        .str.lower()
    )
    return set(zip(df["class_norm"], df["stem"]))


def remap_absolute_path(p: Path) -> Path:
    s = str(p)
    if CROP_PREFIX and CROP_PREFIX_NEW and s.startswith(CROP_PREFIX):
        return Path(CROP_PREFIX_NEW + s[len(CROP_PREFIX) :])
    return p


def build_valid85_sampled_exemplar_table(
    included_txt: Path,
    per_class_csv: Path,
    per_file_csv: Path,
    sampled_csv: Path,
    precision_threshold: float,
) -> pd.DataFrame:
    included = set(load_included_categories(included_txt))
    valid_classes = load_valid_classes(per_class_csv, precision_threshold)
    eligible_cats = included & valid_classes
    valid_pairs = load_valid_pairs(per_file_csv, precision_threshold)

    sampled = pd.read_csv(sampled_csv)
    sampled = sampled[sampled["trial_type"] == "regular"].copy()
    sampled["category"] = sampled["category"].astype(str).str.strip().str.lower()
    sampled["stem"] = sampled["stem"].astype(str).str.strip().str.lower()

    mask = sampled.apply(lambda r: (r["category"], r["stem"]) in valid_pairs, axis=1)
    sampled = sampled.loc[mask].copy()
    sampled = sampled[sampled["category"].isin(eligible_cats)].copy()
    return sampled[["category", "path", "stem"]].reset_index(drop=True)


def collect_things_npy_paths_per_model(
    embed_root: Path,
    allowed_categories_lower: set[str],
    model_label: str,
) -> tuple[dict[str, list[Path]], dict]:
    """All ``*.npy`` under each category folder (recursive), sorted by path string."""
    out: dict[str, list[Path]] = {}
    stats = {
        "model": model_label,
        "n_categories_requested": len(allowed_categories_lower),
        "n_categories_with_subdir": 0,
        "n_npy_files": 0,
    }
    for cat in sorted(allowed_categories_lower):
        cdir = resolve_category_subdir(embed_root, cat)
        if cdir is None:
            out[cat] = []
            continue
        stats["n_categories_with_subdir"] += 1
        paths = sorted(
            (remap_absolute_path(p) for p in cdir.rglob("*.npy") if p.is_file()),
            key=lambda x: str(x).lower(),
        )
        stats["n_npy_files"] += len(paths)
        out[cat] = paths
    return out, stats


def zscore_rows(X: np.ndarray, eps: float = 1e-10) -> np.ndarray:
    mu = X.mean(axis=0)
    sig = X.std(axis=0)
    return (X - mu) / (sig + eps)


In [3]:
def load_normalized_npy_vector(path: Path) -> np.ndarray | None:
    p = remap_absolute_path(path)
    if not p.is_file():
        return None
    v = np.load(p, mmap_mode="r")
    v = np.asarray(v, dtype=np.float64).ravel()
    n = np.linalg.norm(v)
    if n > 0:
        v = v / n
    return v


def average_precomputed_embeddings_for_categories(
    paths_by_cat: dict[str, list[Path]],
) -> tuple[list[str], np.ndarray, dict]:
    categories = sorted(paths_by_cat.keys())
    diag: dict = {"missing_paths": [], "n_paths": {}, "skipped_categories": []}
    cat_vectors: list[np.ndarray] = []
    categories_out: list[str] = []

    for cat in categories:
        paths_in_order = list(dict.fromkeys(paths_by_cat[cat]))
        vecs: list[np.ndarray] = []
        miss: list[str] = []
        for p in paths_in_order:
            v = load_normalized_npy_vector(p)
            if v is None:
                miss.append(str(remap_absolute_path(p)))
            else:
                vecs.append(v)
        diag["missing_paths"].extend(miss[:20])
        diag["n_paths"][cat] = {
            "total": len(paths_in_order),
            "unique_paths": len(paths_in_order),
            "found": len(vecs),
        }
        if not vecs:
            diag["skipped_categories"].append(cat)
            continue
        mean_vec = np.mean(np.stack(vecs, axis=0), axis=0)
        cat_vectors.append(mean_vec)
        categories_out.append(cat)

    X = np.stack(cat_vectors, axis=0) if cat_vectors else np.zeros((0, 0))
    return categories_out, X, diag


In [4]:
for category_set in CATEGORY_SETS:
    inc_path = CATEGORY_FILES[category_set]
    if category_set == "valid129":
        allowed = set(load_included_categories(inc_path))
        inc_all = sorted(allowed)
        clip_paths_by_cat, st_c = collect_things_npy_paths_per_model(
            THINGS_CLIP_NPY_DIR, allowed, "clip"
        )
        dino_paths_by_cat, st_d = collect_things_npy_paths_per_model(
            THINGS_DINOV3_DIR, allowed, "dinov3"
        )
        clip_paths_by_cat = {c: clip_paths_by_cat.get(c, []) for c in inc_all}
        dino_paths_by_cat = {c: dino_paths_by_cat.get(c, []) for c in inc_all}
        n_clip_files = st_c["n_npy_files"]
        n_dino_files = st_d["n_npy_files"]
        n_cat = len(inc_all)
        n_with_both = sum(
            1 for c in inc_all if clip_paths_by_cat[c] and dino_paths_by_cat[c]
        )
        exemplar_source = "things_per_model_npy_trees_valid129_inclusion_unpaired_avg"
        print(f"\n=== {category_set} (THINGS) ===")
        print("included file:", inc_path)
        print("clip_stats:", st_c)
        print("dinov3_stats:", st_d)
        print("categories with >=1 exemplar in BOTH models:", n_with_both, "/", n_cat)
    elif category_set == "valid85":
        exemplar_df = build_valid85_sampled_exemplar_table(
            inc_path,
            PER_CLASS_PRECISION_CSV,
            PER_FILE_PRECISION_CSV,
            SAMPLED_EXEMPLAR_CSV,
            PRECISION_THRESHOLD,
        )
        cats85 = sorted(set(exemplar_df["category"].astype(str).str.strip().str.lower()))
        allowed = set(cats85)
        clip_paths_by_cat, st_c = collect_things_npy_paths_per_model(
            THINGS_CLIP_NPY_DIR, allowed, "clip"
        )
        dino_paths_by_cat, st_d = collect_things_npy_paths_per_model(
            THINGS_DINOV3_DIR, allowed, "dinov3"
        )
        n_clip_files = st_c["n_npy_files"]
        n_dino_files = st_d["n_npy_files"]
        n_cat = len(cats85)
        n_with_both = sum(
            1 for c in cats85 if clip_paths_by_cat.get(c) and dino_paths_by_cat.get(c)
        )
        exemplar_source = "things_per_model_npy_trees_valid85_category_set_same_as_bv06_unpaired_avg"
        print(f"\n=== {category_set} (THINGS) ===")
        print("included file:", inc_path)
        print("valid85 category count (BV gates):", n_cat)
        print("clip_stats:", st_c)
        print("dinov3_stats:", st_d)
        print("categories with >=1 exemplar in BOTH models:", n_with_both, "/", n_cat)
    else:
        raise ValueError(category_set)

    both = sorted(
        c
        for c in clip_paths_by_cat
        if clip_paths_by_cat[c] and dino_paths_by_cat.get(c)
    )
    clip_paths_by_cat = {c: clip_paths_by_cat[c] for c in both}
    dino_paths_by_cat = {c: dino_paths_by_cat[c] for c in both}

    out_dir = OUT_ROOT / category_set
    out_dir.mkdir(parents=True, exist_ok=True)

    cats_c, Xc_raw, diag_c = average_precomputed_embeddings_for_categories(clip_paths_by_cat)
    cats_d, Xd_raw, diag_d = average_precomputed_embeddings_for_categories(dino_paths_by_cat)
    assert cats_c == cats_d

    Xc_z = zscore_rows(Xc_raw)
    Xd_z = zscore_rows(Xd_raw)

    dim_cols_c = [f"dim_{i}" for i in range(Xc_z.shape[1])]
    dim_cols_d = [f"dim_{i}" for i in range(Xd_z.shape[1])]

    def save_cat_emb(csv_path: Path, cats: list[str], X: np.ndarray, dim_cols: list[str]) -> None:
        df = pd.DataFrame(X, columns=dim_cols)
        df.insert(0, "category", cats)
        df.to_csv(csv_path, index=False)

    save_cat_emb(
        out_dir / f"things_clip_exemplar_avg_zscore_within_{category_set}.csv",
        cats_c,
        Xc_z,
        dim_cols_c,
    )
    save_cat_emb(
        out_dir / f"things_dinov3_exemplar_avg_zscore_within_{category_set}.csv",
        cats_d,
        Xd_z,
        dim_cols_d,
    )
    save_cat_emb(
        out_dir / f"things_clip_exemplar_avg_raw_within_{category_set}.csv",
        cats_c,
        Xc_raw,
        dim_cols_c,
    )
    save_cat_emb(
        out_dir / f"things_dinov3_exemplar_avg_raw_within_{category_set}.csv",
        cats_d,
        Xd_raw,
        dim_cols_d,
    )

    meta = {
        "dataset": "things",
        "category_set": category_set,
        "exemplar_source": exemplar_source,
        "averaging_note": "CLIP and DINOv3 file basenames differ (embedding_* vs category_*); means are taken over each model's .npy set per category, restricted to categories with >=1 exemplar in both trees.",
        "included_categories_txt": str(inc_path),
        "n_categories": len(cats_c),
        "n_clip_npy_files_total": n_clip_files,
        "n_dinov3_npy_files_total": n_dino_files,
        "n_inclusion_or_valid85_categories": n_cat,
        "n_categories_with_at_least_one_exemplar_in_both_models": n_with_both,
        "precision_threshold": PRECISION_THRESHOLD,
        "things_clip_npy_dir": str(THINGS_CLIP_NPY_DIR),
        "things_dinov3_embeddings_dir": str(THINGS_DINOV3_DIR),
        "clip_stats": st_c,
        "dinov3_stats": st_d,
        "categories_alphabetical": cats_c,
        "diagnostics_clip": {k: v for k, v in diag_c.items() if k != "missing_paths"},
        "diagnostics_dinov3": {k: v for k, v in diag_d.items() if k != "missing_paths"},
        "n_missing_paths_reported_clip": len(diag_c.get("missing_paths", [])),
        "n_missing_paths_reported_dino": len(diag_d.get("missing_paths", [])),
    }
    (out_dir / "things_exemplar_embedding_run.json").write_text(json.dumps(meta, indent=2))
    print("Wrote:", out_dir / f"things_clip_exemplar_avg_zscore_within_{category_set}.csv")
    print("Wrote:", out_dir / f"things_dinov3_exemplar_avg_zscore_within_{category_set}.csv")
    print("Wrote:", out_dir / "things_exemplar_embedding_run.json")

print("\nDone.")



=== valid129 (THINGS) ===
included file: /home/j7yang/babyview-projects/vss2026/object-detection/data/included_categories_valid129.txt
clip_stats: {'model': 'clip', 'n_categories_requested': 129, 'n_categories_with_subdir': 129, 'n_npy_files': 1845}
dinov3_stats: {'model': 'dinov3', 'n_categories_requested': 129, 'n_categories_with_subdir': 129, 'n_npy_files': 1845}
categories with >=1 exemplar in BOTH models: 129 / 129
Wrote: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/manuscript-2026/exemplar_set_embeddings/valid129/things_clip_exemplar_avg_zscore_within_valid129.csv
Wrote: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/manuscript-2026/exemplar_set_embeddings/valid129/things_dinov3_exemplar_avg_zscore_within_valid129.csv
Wrote: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/manuscript-2026/exemplar_set_embeddings/valid129/things_exemplar_embedding_run.json

=== valid85 (THINGS) ===
included file: /home/j7yang/babyview-projects